# 13 - Splink Demonstration (train -> predict -> cluster -> evaluate)

Notebook end-to-end yang memakai adapter `src/splink_pipeline.py` dan CLI
`src/splink_cli.py` untuk menjalankan dedupe probabilistic dengan Splink.

Label: 107 hasil manual review (`manual_review_queue.csv`) + 300 near-miss
negatif auto-label (hanya setuju pada DOB, semua field identitas beda) dari
`scripts/build_reviewed_labels.py`. Total 407 label.

Perbaikan training: estimasi `u` dengan random sampling lebih besar,
estimasi prior `probability_two_random_records_match`, dan `m` dari pairwise labels.

Guard identitas: prediksi positif membutuhkan setidaknya N field identitas
(email/phone/nama) yang setuju secara eksak, karena Splink over-confident
terhadap pasangan yang hanya berbagi field lemah (dob/city).

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    display
except NameError:
    def display(x):
        print(x.to_string() if hasattr(x, 'to_string') else x)

import pandas as pd
from importlib.util import find_spec

assert find_spec('splink'), 'pip install -e ".[splink]"'

from src.splink_pipeline import (
    prepare_splink_input,
    train_splink_pipeline,
    cluster_predictions,
    evaluate_splink_predictions,
    tune_splink_threshold,
    load_reviewed_labels,
)
from src.io import load_customer_csv

INPUT = Path('data/raw/crm_50000_customers_dirty_v3.csv')
LABELS = Path('data/processed/splink_reviewed_labels_v2.csv')
OUT_DIR = Path('data/processed/splink_demo')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('input:', INPUT.exists(), '| labels (107+300):', LABELS.exists())

## 1. Bridge label manual + near-miss otomatis

`load_reviewed_labels` membaca queue semicolon dan / atau format Splink `record_id_l,record_id_r,clerical_match_score` tanpa memakai `customer_id`.

In [ ]:
labels = load_reviewed_labels(LABELS)
print('label shape:', labels.shape)
print('distribusi score:\n', labels['clerical_match_score'].value_counts())
print(labels.head())

## 2. Train m/u dan predict pairwise

Urutan: `u` random sampling (5e6) -> prior `probability_two_random_records_match`
-> `m` dari pairwise labels. WARNING bahwa sebagian level Levenshtein
`name_key`/`address` tidak teramati di label adalah ekspektasi normal.

In [ ]:
predictions = train_splink_pipeline(INPUT, LABELS, OUT_DIR / 'splink_predictions.csv')
print('predictions:', len(predictions), 'pairs')
print(predictions['match_probability'].describe())

## 3. Evaluasi di reviewed labels (107) + guard identitas

Metrik dihitung di sample label yang di-review manusia. Guard = berapa
field identitas (email/phone/nama) harus setuju secara eksak di samping
probabilitas.

In [ ]:
std = prepare_splink_input(load_customer_csv(INPUT))
tune_plain, summary_plain = tune_splink_threshold(predictions, LABELS, thresholds=[0.3, 0.5, 0.7, 0.9])
display(summary_plain[['threshold', 'true_positive', 'false_positive', 'precision', 'recall', 'f1']])

In [ ]:
rows = []
for guard in [0, 1, 2]:
    for thr in [0.3, 0.5, 0.7]:
        m = evaluate_splink_predictions(predictions, LABELS, threshold=thr, standardized=std, min_identity_agreement=guard)
        rows.append({'guard': guard, 'threshold': thr, 'TP': m['true_positive'], 'FP': m['false_positive'], 'F1': m['f1']})
guard_table = pd.DataFrame(rows)
display(guard_table.pivot(index='threshold', columns='guard', values=['FP', 'F1']))

## 4. Cluster pairwise -> entity (gunakan guard=2)

Guard=2 menghilangkan false positive yang hanya menyetujui dob/city.

In [ ]:
clusters = cluster_predictions(std, predictions, threshold=0.7)

print('records:', len(clusters), '| clusters:', clusters['cluster_id'].nunique())
print('dedupe merges:', len(clusters) - clusters['cluster_id'].nunique())

## 5. Perbandingan: baseline deterministic vs Splink

Rule agreement-count dari pipeline `src` dievaluasi di label yang sama.

In [ ]:
from src.pipeline import run_candidate_pipeline
base = run_candidate_pipeline(INPUT, OUT_DIR / 'baseline_predictions.csv', log_level='WARNING')
base['pair_key'] = base.apply(lambda r: (min(int(r['left_row_index']), int(r['right_row_index'])), max(int(r['left_row_index']), int(r['right_row_index']))), axis=1)
base = base.drop_duplicates('pair_key')
lab = pd.read_csv(LABELS)
lab['pair_key'] = lab.apply(lambda r: (min(int(r['record_id_l']), int(r['record_id_r'])), max(int(r['record_id_l']), int(r['record_id_r']))), axis=1)
joined = lab[['pair_key', 'clerical_match_score']].merge(base[['pair_key', 'agreement_count', 'match_decision']], on='pair_key', how='left')
joined['agreement_count'] = joined['agreement_count'].fillna(0)

In [ ]:
comp = pd.DataFrame({'rule': ['baseline>=2', 'baseline>=4', 'splink 0.7 no-guard', 'splink 0.7 guard=2'],
                    'TP': [0,0,0,0], 'FP': [0,0,0,0], 'F1': [0,0,0,0]})
def evalrow(mask):
    tp = int(((joined['clerical_match_score']==1)&mask).sum()); fp = int(((joined['clerical_match_score']==0)&mask).sum())
    fn = int(((joined['clerical_match_score']==1)&~mask).sum())
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    f1 = 2*p*r/(p+r) if p+r else 0
    return tp, fp, f1
for i,(rule,mask) in enumerate([
    ('baseline>=2', joined['agreement_count']>=2),
    ('baseline>=4', joined['agreement_count']>=4),
    ('splink 0.7 no-guard', None),
    ('splink 0.7 guard=2', None),
]):
    if rule.startswith('splink'):
        m = evaluate_splink_predictions(predictions, LABELS, threshold=0.7, standardized=std, min_identity_agreement=2 if 'guard=2' in rule else 0)
        comp.loc[i] = [rule, m['true_positive'], m['false_positive'], m['f1']]
    else:
        tp, fp, f1 = evalrow(mask); comp.loc[i] = [rule, tp, fp, f1]
display(comp)

## 6. Artifacts
Semua output tersimpan di `data/processed/splink_demo/`.

In [ ]:
for path in sorted(list(OUT_DIR.glob('splink_*')) + list(OUT_DIR.glob('baseline_*'))):
    print('-', path.name, path.stat().st_size, 'bytes')